In [ ]:
import polars as pl
from ydata_profiling import ProfileReport

DATA_DIR = "data"
CLEANED_PATH = f"{DATA_DIR}/cleaned/cnps_cleaned.parquet"
ANSTAT_PATH = "REQUETES ANSTAT_MODULE EMPLOYEURS.xlsx"

## 1. Chargement de la base nettoyée (`cnps_cleaned.parquet`)

In [ ]:
df = pl.read_parquet(CLEANED_PATH)

print(f"Lignes: {df.height:,}")
print(f"Colonnes: {df.width}")
df.head(10)

In [ ]:
# Les Valeurs manquantes et types par colonne
df.null_count().transpose(include_header=True, header_name="colonne", column_names=["n_null"]).with_columns(
    (pl.col("n_null") / df.height * 100).round(2).alias("pct_null")
).sort("pct_null", descending=True)

## 2. Rapport ydata-profiling

27,5M lignes est trop volumineux pour un profiling exhaustif (temps de calcul, mémoire). On échantillonne un sous-ensemble représentatif pour obtenir un rapport exploitable rapidement — à ajuster (`SAMPLE_SIZE`) selon le temps de calcul acceptable.

In [ ]:
SAMPLE_SIZE = 200_000

df_sample = df.sample(n=SAMPLE_SIZE, seed=42).to_pandas()

profile = ProfileReport(
    df_sample,
    title="CNPS - Exploration cnps_cleaned (échantillon)",
    minimal=True,  # désactive les calculs les plus coûteux (corrélations lourdes, interactions)
)
profile.to_notebook_iframe()

In [ ]:
# Sauvegarde du rapport en HTML pour consultation/partage
profile.to_file("docs/profiling_cnps_cleaned.html")

## 3. Focus sur `SECTEUR_ACTIVITE` (nomenclature CNPS)

In [ ]:
# Répartition des effectifs et du salaire moyen par secteur (nomenclature CNPS)
secteur_stats = (
    df.group_by("SECTEUR_ACTIVITE")
    .agg(
        pl.len().alias("n_lignes"),
        pl.col("ID_EMPLOYEUR").n_unique().alias("n_entreprises"),
        pl.col("ID_INDIV").n_unique().alias("n_individus"),
        pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"),
        pl.col("SALAIRE_BRUT_MENS").median().round(0).alias("salaire_median"),
    )
    .sort("n_lignes", descending=True)
)
secteur_stats

## 4. Chargement du fichier ANSTAT (`REQUETES ANSTAT_MODULE EMPLOYEURS.xlsx`)

Référentiel entreprises (feuille `DATA`) : raison sociale, secteur d'activité (nomenclature CEPICI, différente de la nomenclature CNPS), forme juridique, date de début d'activité, motif de radiation, etc. La feuille `SQL` ne contient que la requête source, pas de données à charger.

In [ ]:
df_anstat = pl.read_excel(ANSTAT_PATH, sheet_name="DATA")

print(f"Lignes: {df_anstat.height:,}")
print(f"Colonnes: {df_anstat.width}")
df_anstat.head(10)

In [ ]:
# Valeurs manquantes et secteurs distincts côté ANSTAT
print("N secteurs distincts (ANSTAT):", df_anstat["SECTEUR_ACTIVITE"].n_unique())
print("Pct SECTEUR_ACTIVITE manquant:", round(df_anstat["SECTEUR_ACTIVITE"].null_count() / df_anstat.height * 100, 2), "%")
print()
df_anstat["SECTEUR_ACTIVITE"].value_counts().sort("count", descending=True).head(20)

## 5. Test de jointure CNPS ↔ ANSTAT sur `RAISON_SOCIALE`

Il n'y a pas d'identifiant entreprise commun entre les deux fichiers (le `NUMERO_DFE`/`NUMERO_RCCM` de l'ANSTAT n'a pas d'équivalent direct confirmé côté CNPS) : la seule clé candidate est le nom d'entreprise. On normalise la raison sociale des deux côtés (majuscules, espaces, ponctuation) avant de mesurer le taux de correspondance.

In [ ]:
def normalize_raison_sociale(col: pl.Expr) -> pl.Expr:
    return (
        col.str.to_uppercase()
        .str.replace_all(r"[^A-Z0-9 ]", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )

cnps_firms = (
    df.select("ID_EMPLOYEUR", "RAISON_SOCIALE")
    .unique()
    .with_columns(normalize_raison_sociale(pl.col("RAISON_SOCIALE")).alias("RAISON_SOCIALE_NORM"))
)

anstat_firms = df_anstat.with_columns(
    normalize_raison_sociale(pl.col("RAISON_SOCIALE")).alias("RAISON_SOCIALE_NORM")
)

print("Entreprises CNPS uniques:", cnps_firms.height)
print("Entreprises ANSTAT uniques (lignes):", anstat_firms.height)

In [ ]:
# Taux de correspondance (match exact sur raison sociale normalisée)
# suffixe "_anstat" sur les colonnes ANSTAT en collision de nom (RAISON_SOCIALE, SECTEUR_ACTIVITE)
matched = cnps_firms.join(anstat_firms, on="RAISON_SOCIALE_NORM", how="inner", suffix="_anstat")

n_matched = matched["ID_EMPLOYEUR"].n_unique()
n_total = cnps_firms["ID_EMPLOYEUR"].n_unique()

print(f"Entreprises CNPS matchées: {n_matched:,} / {n_total:,} ({n_matched / n_total * 100:.1f}%)")
matched.select("RAISON_SOCIALE", "RAISON_SOCIALE_NORM", "SECTEUR_ACTIVITE_anstat", "FORME JURIDIQUE").head(20)

In [ ]:
# Aperçu des entreprises CNPS non matchées (pour juger s'il faut un matching approché)
unmatched = cnps_firms.join(anstat_firms, on="RAISON_SOCIALE_NORM", how="anti")
print(f"Non matchées: {unmatched.height:,} ({unmatched.height / n_total * 100:.1f}%)")
unmatched.select("RAISON_SOCIALE").sample(n=20, seed=42)